In [10]:
import math

class Value:
    def __init__(self, data, _children=()):
        self.data = data
        self.grad = 0.0
        self._prev = set(_children)
        self._backward = lambda: None
        
    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self,))

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def __neg__(self):         
        return self * -1

    def __sub__(self, other):       
        return self + (-other)

    def __radd__(self, other):      # other + self
        return self + other

    def __rmul__(self, other):      # other * self
        return self * other

    def __rsub__(self, other):      # other - self
        return other + (-self)

    def __pow__(self, other):    
        assert isinstance(other, (int, float))
        out = Value(self.data ** other, (self,))
    
        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward
    
        return out
    def backward(self):
        topo = []

        visited = set()

        def build_topo(v):

            if v not in visited:
                visited.add(v)

            for child in v._prev:
                build_topo(child)

            topo.append(v)

        build_topo(self)

        self.grad = 1.0

        for v in reversed(topo):
            v._backward()

In [11]:
import random

class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()

    def parameters(self):
        return self.w + [self.b]

In [12]:
class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

In [13]:
class MLP:

    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return[p for layer in self.layers for p in layer.parameters()]

In [14]:
xs = [
    [0.0,0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0]
]

ys = [-1.0, 1.0, 1.0, -1.0]

n = MLP(2, [4,1])

for k in range(200):
    ypred = []

    for x in xs:
        prediction = n(x)
        ypred.append(prediction)

    loss = None

    for true_answer, prediction in zip(ys, ypred):
        error = prediction - true_answer
        squared_error = error ** 2

        if loss is None:
            loss = squared_error
        else:
            loss = loss + squared_error

    for p in n.parameters():
        p.grad = 0.0
    loss.backward()

    learning_rate = 0.05

    for parameter in n.parameters():
        old_value = parameter.data
        gradient = parameter.data

        new_value = old_value - learning_rate * gradient

        parameter.data = new_value

    print(k, loss.data)

    

0 7.26091229534648
1 7.1542314407718735
2 7.0354596056834255
3 6.904605762328202
4 6.762185621611489
5 6.609271163225593
6 6.447489440005881
7 6.278963815910093
8 6.106200334506585
9 5.931931890384135
10 5.758940519317021
11 5.589881336525879
12 5.427129806203579
13 5.272668073220855
14 5.128018023824937
15 4.994220758220942
16 4.871855925781235
17 4.761090701700979
18 4.66174705851085
19 4.573376845907048
20 4.495336258306594
21 4.426853802879245
22 4.367088328320787
23 4.315175708211999
24 4.270264261382142
25 4.2315399430985225
26 4.198242841197497
27 4.169676676223863
28 4.14521294789705
29 4.1242911866088106
30 4.106416528568833
31 4.091155583645755
32 4.078131333736287
33 4.067017600421604
34 4.057533458174127
35 4.049437842287203
36 4.042524504927907
37 4.036617403031071
38 4.031566552857937
39 4.027244353123727
40 4.023542357577924
41 4.020368465504691
42 4.017644492232515
43 4.015304079461189
44 4.01329090560206
45 4.011557158356941
46 4.01006223471358
47 4.008771636920485
48 